# Explicit quantum realization for the class-29 siglocal facet

## Purpose

This notebook searches for an **explicit tripartite quantum realization**
violating the class-29 facet obtained by the white-noise-normalized LP.
The inequality is written as

$$
I[p]=\sum_{a,b,c,x,y,z}c_{abc|xyz}\,p(abc|xyz)\geq\beta.
$$

In the primitive-integer normalization used here, the siglocal bound is

$$
\beta=-36.
$$

The equivalent normalization used by the preceding LP and NPA notebooks
multiplies the entire inequality by $8/135$, giving
$\beta=-32/15$.

The notebook has four safeguards:

1. it embeds the exact integer coefficients and makes the
   $p(abc|xyz)$ coordinate order explicit;
2. it regenerates the corrected 8128 siglocal vertices and checks the
   bound and facet dimension;
3. it uses a see-saw whose state step and binary-measurement steps are
   solved exactly by Hermitian eigendecompositions—there is no SDP
   solver tolerance inside the see-saw;
4. it reconstructs all 64 probabilities from the final state and
   measurements and evaluates the inequality again independently.

The output is a feasible quantum strategy, not merely an outer NPA
relaxation. The see-saw remains a nonconvex local search, so it does not
by itself prove global optimality. Multiple random restarts are used to
address this practical limitation.

## Why three qubits and projective measurements are sufficient here

Each party has two binary measurements. For a fixed state and the other
parties' measurements, a local binary POVM has the form

$$
\{E_{0|x},E_{1|x}\},\qquad
E_{1|x}=\mathbb I-E_{0|x},\qquad 0\preceq E_{0|x}\preceq\mathbb I.
$$

Its contribution to the objective can be written as

$$
\operatorname{tr}(K_{1|x})+
\operatorname{tr}\!\left[E_{0|x}(K_{0|x}-K_{1|x})\right].
$$

Therefore, the exact minimizer is the projector onto the negative
eigenspace of $K_{0|x}-K_{1|x}$. This is precisely the solution that a
local SDP would return, but it can be obtained directly and more
accurately by diagonalizing a $2\times2$ Hermitian matrix.

Moreover, with two dichotomic observables per party, Jordan's lemma
reduces the optimization to blocks of local dimension at most two.
Since the Bell functional is linear, one block attains the optimum.
Thus a pure three-qubit state and binary projective measurements are
sufficient for this scenario.

## 1. Standard-library setup

No external Julia package is required by this notebook. The calculation
uses only Julia's standard libraries. It was written for the Julia 1.12
environment used by the preceding project notebooks.

In [4]:
using LinearAlgebra
using Random
using Printf
using Dates
using SHA
using Serialization

println("Julia version: ", VERSION)

Julia version: 1.12.6


## 2. Canonical probability coordinates and exact inequality

The 64-vector is split into input blocks ordered as

$$
xyz=000,001,010,011,100,101,110,111,
$$

and each block uses the output order

$$
abc=000,001,010,011,100,101,110,111.
$$

Hence $z$ and $c$ are the fastest-changing bits, and Julia's one-based
coordinate for $p(abc|xyz)$ is

$$
1+c+2b+4a+8z+16y+32x.
$$

The integer coefficients below are the exact primitive form of the
class-29 row in `primitive_integer_coefficients.tsv`. The code never
reshapes this vector to perform a calculation; every access goes through
`prob_index`, avoiding Julia column-major-order ambiguities.

In [6]:
const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)

const CLASS = 29
const BETA_PRIMITIVE = -36
const NORMALIZATION_SCALE = 8 // 135
const BETA_NORMALIZED = BETA_PRIMITIVE * NORMALIZATION_SCALE

const C_PRIMITIVE = Int[
    -3, 5, 1, 9, -8, 0, -4, -5, -3, -4, 2, -5, 11, 4, 1, 0, 0, 1, 0, -5, 14, 0, 8, 0, -1, 0, -1, 0, -9, 1, 0, 1, -5, -4, 0, -14, 5, 0, 4, -4, -1, 0, 0, 10, -11, -10, -1, 0, -14, -9, -14, 0, -24, -10, -24, -10, 0, -10, -9, -10, 1, 0, 1, 0
]

@assert length(C_PRIMITIVE) == 64
@assert BETA_NORMALIZED == -32 // 15

# Human-readable 8-by-8 display only: rows are xyz and columns are abc.
coefficient_table = [
    C_PRIMITIVE[prob_index(a, b, c, x, y, z)]
    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
]

coefficient_bytes = Vector{UInt8}(codeunits(join(string.(C_PRIMITIVE), ",")))
coefficient_digest = bytes2hex(sha256(coefficient_bytes))

println("Class:                    ", CLASS)
println("Primitive siglocal bound: ", BETA_PRIMITIVE)
println("Normalized bound:         ", BETA_NORMALIZED)
println("Coefficient digest:       ", coefficient_digest)
println("\nCoefficient table (rows xyz; columns abc):")
display(coefficient_table)

Class:                    29
Primitive siglocal bound: -36
Normalized bound:         -32//15
Coefficient digest:       d7e0fcc597d594c4584fb5a6e6cfc10b06eb731073bc98df8660a33db1262094

Coefficient table (rows xyz; columns abc):


8×8 Matrix{Int64}:
  -3    5    1    9   -8    0   -4   -5
  -3   -4    2   -5   11    4    1    0
   0    1    0   -5   14    0    8    0
  -1    0   -1    0   -9    1    0    1
  -5   -4    0  -14    5    0    4   -4
  -1    0    0   10  -11  -10   -1    0
 -14   -9  -14    0  -24  -10  -24  -10
   0  -10   -9  -10    1    0    1    0

## 3. Independent siglocal validity and facet audit

This section reproduces the corrected cycle-polytope construction. The
clockwise deterministic behaviors obey

$$
a=f_A(x,z),\qquad b=f_B(y,x),\qquad c=f_C(z,y),
$$

whereas the counterclockwise behaviors obey

$$
a=f_A(x,y),\qquad b=f_B(y,z),\qquad c=f_C(z,x).
$$

Their union contains 8128 distinct vertices. The coefficient audit is
exact because both the vertices and the inequality are integer-valued.
Only the affine-rank calculation uses floating-point linear algebra.

In [8]:
# Evaluate one of the 16 deterministic binary functions of two bits.
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

# Construct one deterministic behavior in canonical coordinates.
function deterministic_vertex(fA::Int, fB::Int, fC::Int, orientation::Symbol)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1
    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end
    return vertices
end

function affine_dimension(points; atol=1e-9)
    number_of_points = size(points, 1)
    number_of_points == 0 && return -1
    number_of_points == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

@assert size(vertices_clockwise) == (4096, 64)
@assert size(vertices_counterclockwise) == (4096, 64)
@assert size(vertices_cycle) == (8128, 64)

primitive_values = [
    sum(C_PRIMITIVE[index] * Int(vertices_cycle[row, index]) for index in 1:64)
    for row in axes(vertices_cycle, 1)
]

@assert minimum(primitive_values) == BETA_PRIMITIVE
@assert all(value -> value >= BETA_PRIMITIVE, primitive_values)

saturating_indices = findall(==(BETA_PRIMITIVE), primitive_values)
polytope_dimension = affine_dimension(vertices_cycle)
face_dimension = affine_dimension(vertices_cycle[saturating_indices, :])

@assert length(saturating_indices) == 270
@assert polytope_dimension == 50
@assert face_dimension == 49

uniform_primitive_value = sum(C_PRIMITIVE) // 8
@assert (uniform_primitive_value - BETA_PRIMITIVE) * NORMALIZATION_SCALE == 1

println("Corrected cycle polytope")
println("  distinct vertices:       ", size(vertices_cycle, 1))
println("  affine dimension:        ", polytope_dimension)
println("  exact minimum:           ", minimum(primitive_values))
println("  saturating vertices:     ", length(saturating_indices))
println("  saturating face dim.:    ", face_dimension)
println("  facet:                   ", face_dimension == polytope_dimension - 1)
println("  uniform-behavior value: ", uniform_primitive_value)

Corrected cycle polytope
  distinct vertices:       8128
  affine dimension:        50
  exact minimum:           -36
  saturating vertices:     270
  saturating face dim.:    49
  facet:                   true
  uniform-behavior value: -153//8


## 4. Quantum realization and see-saw primitives

A realization consists of a normalized vector
$|\psi\rangle\in(\mathbb C^2)^{\otimes3}$ and projectors
$M^A_{a|x}$, $M^B_{b|y}$, and $M^C_{c|z}$. Its Bell operator is

$$
W=\sum_{a,b,c,x,y,z}c_{abc|xyz}
M^A_{a|x}\otimes M^B_{b|y}\otimes M^C_{c|z}.
$$

For fixed measurements, the optimal pure state is an eigenvector of
$W$ associated with its smallest eigenvalue.

`effective_operator` deliberately reconstructs each local cost matrix
from matrix units and full tensor products. This is slightly less terse
than a hand-written partial trace, but it makes the subsystem order
explicit and avoids silent row-major/column-major tensor mistakes.

In [10]:
const LOCAL_DIMENSION = 2
const GLOBAL_DIMENSION = LOCAL_DIMENSION^3
const IDENTITY_2 = Matrix{ComplexF64}(I, 2, 2)

tensor3(A, B, C) = kron(kron(A, B), C)

function random_projective_measurement(rng)
    measurements = Vector{Vector{Matrix{ComplexF64}}}(undef, 2)
    for input in 1:2
        vector = randn(rng, 2) + im * randn(rng, 2)
        vector ./= norm(vector)
        outcome_zero = vector * vector'
        outcome_one = IDENTITY_2 - outcome_zero
        measurements[input] = [outcome_zero, outcome_one]
    end
    return measurements
end

random_measurements(rng) = [
    random_projective_measurement(rng)
    for _ in 1:3
]

function bell_operator(measurements, coefficients=C_PRIMITIVE)
    MA, MB, MC = measurements
    operator = zeros(ComplexF64, GLOBAL_DIMENSION, GLOBAL_DIMENSION)

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        coefficient = coefficients[prob_index(a, b, c, x, y, z)]
        iszero(coefficient) && continue

        operator .+= coefficient .* tensor3(
            MA[x + 1][a + 1],
            MB[y + 1][b + 1],
            MC[z + 1][c + 1],
        )
    end

    return Matrix(Hermitian((operator + operator') / 2))
end

function optimize_state(measurements, coefficients=C_PRIMITIVE)
    decomposition = eigen(Hermitian(bell_operator(measurements, coefficients)))
    best_index = argmin(decomposition.values)
    value = Float64(decomposition.values[best_index])
    state = Vector{ComplexF64}(decomposition.vectors[:, best_index])
    state ./= norm(state)
    return value, state
end

# Insert a local 2-by-2 operator into the requested tensor factor.
# For party 1, `other_first` and `other_second` are Bob's and Charlie's
# operators. For party 2 they are Alice's and Charlie's; for party 3 they
# are Alice's and Bob's.
function embed_local(local_operator, party::Int, other_first, other_second)
    party == 1 && return tensor3(local_operator, other_first, other_second)
    party == 2 && return tensor3(other_first, local_operator, other_second)
    party == 3 && return tensor3(other_first, other_second, local_operator)
    error("party must be 1, 2, or 3")
end

# Return H such that tr(rho * embed_local(E,...)) = tr(E*H).
# The identity is reconstructed from the four local matrix units. The
# final Hermitian symmetrization removes only roundoff-level asymmetry.
function effective_operator(rho, party::Int, other_first, other_second)
    effective = zeros(ComplexF64, 2, 2)

    for row in 1:2, column in 1:2
        matrix_unit = zeros(ComplexF64, 2, 2)
        matrix_unit[row, column] = 1
        full_operator = embed_local(
            matrix_unit,
            party,
            other_first,
            other_second,
        )

        # If E[row,column]=1, then tr(E*H)=H[column,row].
        effective[column, row] = tr(rho * full_operator)
    end

    return Matrix(Hermitian((effective + effective') / 2))
end

function local_cost_operators(
    rho,
    measurements,
    party::Int,
    input::Int,
    coefficients=C_PRIMITIVE,
)
    @assert party in 1:3
    @assert input in 0:1

    MA, MB, MC = measurements
    costs = [zeros(ComplexF64, 2, 2) for _ in 1:2]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        local_input = party == 1 ? x : party == 2 ? y : z
        local_input == input || continue

        coefficient = coefficients[prob_index(a, b, c, x, y, z)]
        iszero(coefficient) && continue

        local_output = party == 1 ? a : party == 2 ? b : c

        if party == 1
            environment = effective_operator(
                rho, 1, MB[y + 1][b + 1], MC[z + 1][c + 1],
            )
        elseif party == 2
            environment = effective_operator(
                rho, 2, MA[x + 1][a + 1], MC[z + 1][c + 1],
            )
        else
            environment = effective_operator(
                rho, 3, MA[x + 1][a + 1], MB[y + 1][b + 1],
            )
        end

        costs[local_output + 1] .+= coefficient .* environment
    end

    return [Matrix(Hermitian((cost + cost') / 2)) for cost in costs]
end

# Optimize both binary measurements of one party exactly.
function optimize_party!(
    measurements,
    state,
    party::Int,
    coefficients=C_PRIMITIVE;
    eigenvalue_tolerance=1e-12,
)
    rho = state * state'
    optimized = Vector{Vector{Matrix{ComplexF64}}}(undef, 2)

    for input in 0:1
        K0, K1 = local_cost_operators(
            rho,
            measurements,
            party,
            input,
            coefficients,
        )

        difference = Hermitian((K0 - K1 + (K0 - K1)') / 2)
        decomposition = eigen(difference)

        outcome_zero = zeros(ComplexF64, 2, 2)
        for index in eachindex(decomposition.values)
            if decomposition.values[index] < -eigenvalue_tolerance
                eigenvector = decomposition.vectors[:, index]
                outcome_zero .+= eigenvector * eigenvector'
            end
        end

        outcome_zero = Matrix(Hermitian((outcome_zero + outcome_zero') / 2))
        outcome_one = IDENTITY_2 - outcome_zero
        optimized[input + 1] = [outcome_zero, outcome_one]
    end

    measurements[party] = optimized
    return measurements
end

function quantum_behavior(state, measurements)
    MA, MB, MC = measurements
    probabilities = zeros(Float64, 64)

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        probability_operator = tensor3(
            MA[x + 1][a + 1],
            MB[y + 1][b + 1],
            MC[z + 1][c + 1],
        )
        probabilities[prob_index(a, b, c, x, y, z)] =
            real(dot(state, probability_operator * state))
    end

    return probabilities
end

function bell_value_from_behavior(probabilities, coefficients=C_PRIMITIVE)
    return sum(
        Float64(coefficients[index]) * probabilities[index]
        for index in eachindex(coefficients)
    )
end

bell_value_from_behavior (generic function with 2 methods)

## 5. Regression tests for tensor order and objective evaluation

These tests are independent of the class-29 optimum. They verify that
each effective local operator reproduces the corresponding full
three-qubit trace, and that evaluating a random strategy through its 64
probabilities agrees with evaluating its Bell operator.

In [12]:
function run_internal_regression_tests(; seed=771923)
    rng = MersenneTwister(seed)
    measurements = random_measurements(rng)
    state = randn(rng, 8) + im * randn(rng, 8)
    state ./= norm(state)
    rho = state * state'

    for party in 1:3
        local_test = randn(rng, 2, 2) + im * randn(rng, 2, 2)
        local_test = Matrix(Hermitian((local_test + local_test') / 2))

        if party == 1
            other_first = measurements[2][1][1]
            other_second = measurements[3][2][2]
        elseif party == 2
            other_first = measurements[1][1][1]
            other_second = measurements[3][2][2]
        else
            other_first = measurements[1][1][1]
            other_second = measurements[2][2][2]
        end

        effective = effective_operator(
            rho,
            party,
            other_first,
            other_second,
        )
        full_operator = embed_local(
            local_test,
            party,
            other_first,
            other_second,
        )

        full_value = real(tr(rho * full_operator))
        effective_value = real(tr(local_test * effective))
        @assert isapprox(full_value, effective_value; atol=1e-11, rtol=0.0)
    end

    probabilities = quantum_behavior(state, measurements)
    value_from_probabilities = bell_value_from_behavior(probabilities)
    value_from_operator = real(dot(
        state,
        bell_operator(measurements) * state,
    ))

    @assert isapprox(
        value_from_probabilities,
        value_from_operator;
        atol=1e-10,
        rtol=0.0,
    )

    return true
end

run_internal_regression_tests()
println("Internal tensor-order and objective regression tests passed.")

Internal tensor-order and objective regression tests passed.


## 6. One see-saw run and multiple random restarts

Every iteration performs the following monotone minimization steps:

1. optimize Alice's two measurements for the current state, Bob, and
   Charlie;
2. optimize Bob's measurements using the newly optimized Alice;
3. optimize Charlie's measurements using the newly optimized Alice and
   Bob;
4. diagonalize the resulting Bell operator and choose its minimum
   eigenvector as the new state.

The default 200 restarts are deliberately conservative. In an
independent implementation, the same value was reached from many random
starts in roughly 10–15 iterations. You may reduce `NUMBER_OF_RESTARTS`
for a quick test, but the final research run should retain at least the
default value.

In [14]:
function seesaw_single_run(
    coefficients=C_PRIMITIVE;
    seed::Int,
    maximum_iterations=300,
    convergence_tolerance=1e-12,
    monotonicity_tolerance=1e-9,
)
    rng = MersenneTwister(seed)
    measurements = random_measurements(rng)
    current_value, state = optimize_state(measurements, coefficients)
    history = Float64[current_value]
    converged = false

    for iteration in 1:maximum_iterations
        previous_value = current_value

        for party in 1:3
            optimize_party!(measurements, state, party, coefficients)
        end

        current_value, state = optimize_state(measurements, coefficients)
        push!(history, current_value)

        current_value <= previous_value + monotonicity_tolerance || error(
            "See-saw objective increased from $previous_value to " *
            "$current_value at iteration $iteration (seed $seed)",
        )

        if iteration >= 3 &&
           abs(current_value - previous_value) <= convergence_tolerance
            converged = true
            break
        end
    end

    return (
        value=current_value,
        state=state,
        measurements=measurements,
        history=history,
        iterations=length(history) - 1,
        converged=converged,
        seed=seed,
    )
end

const NUMBER_OF_RESTARTS = 200
const MAXIMUM_ITERATIONS = 300
const CONVERGENCE_TOLERANCE = 1e-12
const MASTER_SEED = 20260929

function run_restarts(
    coefficients=C_PRIMITIVE;
    number_of_restarts=NUMBER_OF_RESTARTS,
    maximum_iterations=MAXIMUM_ITERATIONS,
    convergence_tolerance=CONVERGENCE_TOLERANCE,
    master_seed=MASTER_SEED,
)
    summaries = NamedTuple[]
    best = nothing
    start_time = time()

    println("Starting $number_of_restarts see-saw restarts...")

    for run in 1:number_of_restarts
        seed = master_seed + run - 1
        result = seesaw_single_run(
            coefficients;
            seed=seed,
            maximum_iterations=maximum_iterations,
            convergence_tolerance=convergence_tolerance,
        )

        push!(summaries, (
            run=run,
            seed=seed,
            value=result.value,
            iterations=result.iterations,
            converged=result.converged,
        ))

        if best === nothing || result.value < best.value
            best = merge(result, (run=run,))
            @printf(
                "  run %3d: new best primitive value = %.12f (normalized %.12f)\n",
                run,
                result.value,
                Float64(NORMALIZATION_SCALE) * result.value,
            )
        elseif run % 25 == 0
            @printf("  completed %3d/%3d restarts\n", run, number_of_restarts)
        end
    end

    return summaries, best, time() - start_time
end

run_summaries, best_result, elapsed_seconds = run_restarts()

println("See-saw search finished in ", round(elapsed_seconds; digits=2), " seconds.")
println("Best run: ", best_result.run, " (seed ", best_result.seed, ")")

Starting 200 see-saw restarts...
  run   1: new best primitive value = -36.049951291008 (normalized -2.136293409838)
  run   4: new best primitive value = -36.049951291008 (normalized -2.136293409838)
  run   5: new best primitive value = -36.049951291008 (normalized -2.136293409838)
  run   9: new best primitive value = -36.049951291008 (normalized -2.136293409838)
  completed  25/200 restarts
  completed  50/200 restarts
  completed  75/200 restarts
  run  91: new best primitive value = -36.049951291008 (normalized -2.136293409838)
  completed 100/200 restarts
  completed 125/200 restarts
  completed 150/200 restarts
  completed 175/200 restarts
  completed 200/200 restarts
See-saw search finished in 3.75 seconds.
Best run: 91 (seed 20261019)


## 7. Independent audit of the best realization

This cell does not trust the objective value stored by the see-saw. It
reconstructs every $p(abc|xyz)$ from the final state and projectors,
evaluates the inequality from those 64 probabilities, and checks:

- state normalization;
- Hermiticity, positivity, completeness, and projectivity of every
  measurement;
- positivity and normalization of the behavior;
- nonsignalling residuals;
- agreement between probability and Bell-operator evaluations;
- a violation comfortably larger than numerical residuals.

In [16]:
function measurement_residuals(measurements)
    hermiticity = 0.0
    completeness = 0.0
    projectivity = 0.0
    minimum_eigenvalue = Inf

    for party in 1:3, input in 1:2
        E0, E1 = measurements[party][input]
        completeness = max(completeness, norm(E0 + E1 - IDENTITY_2))

        for operator in (E0, E1)
            hermiticity = max(hermiticity, norm(operator - operator'))
            projectivity = max(projectivity, norm(operator * operator - operator))
            minimum_eigenvalue = min(
                minimum_eigenvalue,
                minimum(eigvals(Hermitian((operator + operator') / 2))),
            )
        end
    end

    return (
        hermiticity=hermiticity,
        completeness=completeness,
        projectivity=projectivity,
        minimum_eigenvalue=minimum_eigenvalue,
    )
end

function behavior_residuals(probabilities)
    normalization = 0.0
    nonsignalling = 0.0

    for (x, y, z) in TRIPLES
        block_sum = sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        )
        normalization = max(normalization, abs(block_sum - 1))
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        left = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        right = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        nonsignalling = max(nonsignalling, abs(left - right))
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        left = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        right = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        nonsignalling = max(nonsignalling, abs(left - right))
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        left = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        right = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        nonsignalling = max(nonsignalling, abs(left - right))
    end

    return (
        minimum_probability=minimum(probabilities),
        maximum_probability=maximum(probabilities),
        normalization=normalization,
        nonsignalling=nonsignalling,
    )
end

best_probabilities = quantum_behavior(
    best_result.state,
    best_result.measurements,
)

primitive_value_from_probabilities = bell_value_from_behavior(best_probabilities)
primitive_value_from_operator = real(dot(
    best_result.state,
    bell_operator(best_result.measurements) * best_result.state,
))

normalized_value = Float64(NORMALIZATION_SCALE) * primitive_value_from_probabilities
normalized_beta = Float64(BETA_NORMALIZED)
primitive_gap = BETA_PRIMITIVE - primitive_value_from_probabilities
normalized_gap = normalized_beta - normalized_value
white_noise_fraction = normalized_gap / (1 + normalized_gap)

measurement_audit = measurement_residuals(best_result.measurements)
behavior_audit = behavior_residuals(best_probabilities)
state_norm_residual = abs(norm(best_result.state) - 1)
evaluation_residual = abs(
    primitive_value_from_probabilities - primitive_value_from_operator,
)

@assert state_norm_residual <= 1e-12
@assert measurement_audit.hermiticity <= 1e-12
@assert measurement_audit.completeness <= 1e-11
@assert measurement_audit.projectivity <= 1e-10
@assert measurement_audit.minimum_eigenvalue >= -1e-10
@assert behavior_audit.minimum_probability >= -1e-10
@assert behavior_audit.maximum_probability <= 1 + 1e-10
@assert behavior_audit.normalization <= 1e-10
@assert behavior_audit.nonsignalling <= 1e-10
@assert evaluation_residual <= 1e-9

# A large-margin regression check. It is intentionally much looser than
# the expected final digits, while still detecting coordinate mistakes.
const EXPECTED_PRIMITIVE_VALUE = -36.0499512910
@assert isapprox(
    primitive_value_from_probabilities,
    EXPECTED_PRIMITIVE_VALUE;
    atol=5e-7,
    rtol=0.0,
)
@assert primitive_gap > 1e-3

# Previous tight-SCS NPA level-4 output. It is a numerical reference,
# not an assertion: SCS and the see-saw differ at roughly 1e-8 here.
const PREVIOUS_NPA_LEVEL4_NORMALIZED = -2.136293398936689

println("\n", "="^72)
println("BEST EXPLICIT QUANTUM REALIZATION")
println("="^72)
@printf("Primitive siglocal bound:       %.12f\n", Float64(BETA_PRIMITIVE))
@printf("Primitive quantum value:        %.12f\n", primitive_value_from_probabilities)
@printf("Primitive violation gap:        %.12f\n", primitive_gap)
println()
@printf("Normalized siglocal bound:      %.12f\n", normalized_beta)
@printf("Normalized quantum value:       %.12f\n", normalized_value)
@printf("Normalized violation gap:       %.12f\n", normalized_gap)
@printf("White-noise fraction:           %.12f (%.6f%%)\n",
        white_noise_fraction, 100 * white_noise_fraction)
println()
@printf("Previous NPA level-4 value:     %.12f\n", PREVIOUS_NPA_LEVEL4_NORMALIZED)
@printf("See-saw minus NPA level 4:      %.3e\n",
        normalized_value - PREVIOUS_NPA_LEVEL4_NORMALIZED)
println()
@printf("State-norm residual:            %.3e\n", state_norm_residual)
@printf("Measurement completeness:      %.3e\n", measurement_audit.completeness)
@printf("Measurement projectivity:      %.3e\n", measurement_audit.projectivity)
@printf("Minimum measurement eigenvalue: %.3e\n", measurement_audit.minimum_eigenvalue)
@printf("Behavior normalization:        %.3e\n", behavior_audit.normalization)
@printf("Nonsignalling residual:        %.3e\n", behavior_audit.nonsignalling)
@printf("Independent evaluation resid.: %.3e\n", evaluation_residual)
println("="^72)


BEST EXPLICIT QUANTUM REALIZATION
Primitive siglocal bound:       -36.000000000000
Primitive quantum value:        -36.049951291008
Primitive violation gap:        0.049951291008

Normalized siglocal bound:      -2.133333333333
Normalized quantum value:       -2.136293409838
Normalized violation gap:       0.002960076504
White-noise fraction:           0.002951340311 (0.295134%)

Previous NPA level-4 value:     -2.136293398937
See-saw minus NPA level 4:      -1.090e-08

State-norm residual:            0.000e+00
Measurement completeness:      0.000e+00
Measurement projectivity:      2.803e-16
Minimum measurement eigenvalue: -3.469e-17
Behavior normalization:        2.220e-16
Nonsignalling residual:        2.220e-16
Independent evaluation resid.: 0.000e+00


## 8. Save the reproducible realization

The notebook writes five files to `class29_seesaw_results`:

- `class29_seesaw_runs.tsv`: value and convergence data for every restart;
- `class29_best_state.tsv`: full-precision real and imaginary components
  of the three-qubit state;
- `class29_best_measurements.tsv`: every entry of all twelve POVM
  elements;
- `class29_best_behavior.tsv`: the 64 probabilities and their individual
  contributions to the inequality;
- `class29_best_realization.jls`: a Julia-serialized copy of the complete
  in-memory realization;
- `run_metadata.txt`: parameters, values, gaps, and audit residuals.

The TSV files are the portable record. The `.jls` file is convenient
when continuing the calculation in the same Julia environment.

In [18]:
const OUTPUT_DIRECTORY = joinpath(pwd(), "class29_seesaw_results")
mkpath(OUTPUT_DIRECTORY)

function write_run_summaries(path, summaries)
    open(path, "w") do io
        println(io, "run\tseed\tprimitive_value\tnormalized_value\titerations\tconverged")
        for result in summaries
            @printf(
                io,
                "%d\t%d\t%.17g\t%.17g\t%d\t%s\n",
                result.run,
                result.seed,
                result.value,
                Float64(NORMALIZATION_SCALE) * result.value,
                result.iterations,
                string(result.converged),
            )
        end
    end
end

function write_state(path, state)
    open(path, "w") do io
        println(io, "basis_index\ta\tb\tc\treal\timag")
        for a in 0:1, b in 0:1, c in 0:1
            index = 1 + c + 2*b + 4*a
            @printf(
                io,
                "%d\t%d\t%d\t%d\t%.17g\t%.17g\n",
                index,
                a,
                b,
                c,
                real(state[index]),
                imag(state[index]),
            )
        end
    end
end

function write_measurements(path, measurements)
    open(path, "w") do io
        println(io, "party\tinput\toutput\trow\tcolumn\treal\timag")
        party_names = ("A", "B", "C")
        for party in 1:3, input in 0:1, output in 0:1
            operator = measurements[party][input + 1][output + 1]
            for row in 1:2, column in 1:2
                @printf(
                    io,
                    "%s\t%d\t%d\t%d\t%d\t%.17g\t%.17g\n",
                    party_names[party],
                    input,
                    output,
                    row,
                    column,
                    real(operator[row, column]),
                    imag(operator[row, column]),
                )
            end
        end
    end
end

function write_behavior(path, probabilities)
    open(path, "w") do io
        println(io, "index\tx\ty\tz\ta\tb\tc\tprobability\tcoefficient\tcontribution")
        for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
            index = prob_index(a, b, c, x, y, z)
            probability = probabilities[index]
            coefficient = C_PRIMITIVE[index]
            @printf(
                io,
                "%d\t%d\t%d\t%d\t%d\t%d\t%d\t%.17g\t%d\t%.17g\n",
                index,
                x,
                y,
                z,
                a,
                b,
                c,
                probability,
                coefficient,
                coefficient * probability,
            )
        end
    end
end

runs_path = joinpath(OUTPUT_DIRECTORY, "class29_seesaw_runs.tsv")
state_path = joinpath(OUTPUT_DIRECTORY, "class29_best_state.tsv")
measurements_path = joinpath(OUTPUT_DIRECTORY, "class29_best_measurements.tsv")
behavior_path = joinpath(OUTPUT_DIRECTORY, "class29_best_behavior.tsv")
realization_path = joinpath(OUTPUT_DIRECTORY, "class29_best_realization.jls")
metadata_path = joinpath(OUTPUT_DIRECTORY, "run_metadata.txt")

write_run_summaries(runs_path, run_summaries)
write_state(state_path, best_result.state)
write_measurements(measurements_path, best_result.measurements)
write_behavior(behavior_path, best_probabilities)

realization = (
    class=CLASS,
    coefficients=C_PRIMITIVE,
    beta_primitive=BETA_PRIMITIVE,
    normalization_scale=NORMALIZATION_SCALE,
    state=best_result.state,
    measurements=best_result.measurements,
    probabilities=best_probabilities,
    primitive_value=primitive_value_from_probabilities,
    normalized_value=normalized_value,
    seed=best_result.seed,
    run=best_result.run,
)

open(realization_path, "w") do io
    serialize(io, realization)
end

open(metadata_path, "w") do io
    println(io, "Class-29 facet quantum see-saw")
    println(io, "timestamp = ", Dates.now())
    println(io, "julia_version = ", VERSION)
    println(io, "coefficient_digest = ", coefficient_digest)
    println(io, "coordinate_order = xyz blocks and abc within blocks; z,c fastest")
    println(io, "number_of_restarts = ", NUMBER_OF_RESTARTS)
    println(io, "maximum_iterations = ", MAXIMUM_ITERATIONS)
    println(io, "convergence_tolerance = ", CONVERGENCE_TOLERANCE)
    println(io, "master_seed = ", MASTER_SEED)
    println(io, "best_run = ", best_result.run)
    println(io, "best_seed = ", best_result.seed)
    println(io, "best_iterations = ", best_result.iterations)
    println(io, "best_converged = ", best_result.converged)
    println(io, "primitive_beta = ", BETA_PRIMITIVE)
    println(io, "primitive_quantum_value = ", repr(primitive_value_from_probabilities))
    println(io, "primitive_violation_gap = ", repr(primitive_gap))
    println(io, "normalized_beta = ", repr(normalized_beta))
    println(io, "normalized_quantum_value = ", repr(normalized_value))
    println(io, "normalized_violation_gap = ", repr(normalized_gap))
    println(io, "white_noise_fraction = ", repr(white_noise_fraction))
    println(io, "previous_npa_level4 = ", repr(PREVIOUS_NPA_LEVEL4_NORMALIZED))
    println(io, "state_norm_residual = ", repr(state_norm_residual))
    println(io, "measurement_hermiticity_residual = ", repr(measurement_audit.hermiticity))
    println(io, "measurement_completeness_residual = ", repr(measurement_audit.completeness))
    println(io, "measurement_projectivity_residual = ", repr(measurement_audit.projectivity))
    println(io, "minimum_measurement_eigenvalue = ", repr(measurement_audit.minimum_eigenvalue))
    println(io, "minimum_probability = ", repr(behavior_audit.minimum_probability))
    println(io, "maximum_probability = ", repr(behavior_audit.maximum_probability))
    println(io, "behavior_normalization_residual = ", repr(behavior_audit.normalization))
    println(io, "nonsignalling_residual = ", repr(behavior_audit.nonsignalling))
    println(io, "independent_evaluation_residual = ", repr(evaluation_residual))
    println(io, "elapsed_seconds = ", repr(elapsed_seconds))
end

println("\nSaved reproducible outputs to: ", abspath(OUTPUT_DIRECTORY))
for path in (
    runs_path,
    state_path,
    measurements_path,
    behavior_path,
    realization_path,
    metadata_path,
)
    println("  ", basename(path))
end


Saved reproducible outputs to: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\class29_seesaw_results
  class29_seesaw_runs.tsv
  class29_best_state.tsv
  class29_best_measurements.tsv
  class29_best_behavior.tsv
  class29_best_realization.jls
  run_metadata.txt


## Interpretation

A successful run should return approximately

$$
I_{\mathrm{quantum}}\approx -36.0499512910<-36
$$

in primitive units, or

$$
I_{\mathrm{quantum}}\approx -2.13629340984
<-\frac{32}{15}
$$

in the white-noise normalization. The expected normalized violation is
approximately $0.00296008$, corresponding to approximately $0.2951\%$
white-noise tolerance.

This is an **explicit numerical quantum violation of a facet** of the
siglocal polytope: the state and every local projector are saved, and
the final value is recomputed directly from their probabilities.

The close agreement with the preceding level-4 NPA calculation strongly
suggests that the see-saw has reached the quantum optimum. Nevertheless,
neither a floating-point see-saw nor an SCS-based NPA calculation is a
formal exact proof of the optimum. If a rigorous paper-level numerical
certificate is required later, the saved realization can be subjected
to interval-arithmetic verification and the NPA calculation can be
repeated with a higher-precision SDP solver.